# 03 - Transform and export

This notebook shows the difference between returned rendered text and an explicit disk write. The target format can express only part of the source calculation model; XYZ is used here for a coordinate-focused example.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from molop import AutoParser, molopconfig

molopconfig.quiet()
sample_candidates = (
    Path("docs/assets/examples/water_mp2.out"),
    Path("../../assets/examples/water_mp2.out"),
    Path("water_mp2.out"),
)
sample_path = next((path for path in sample_candidates if path.is_file()), None)
if sample_path is None:
    raise FileNotFoundError("Place water_mp2.out beside the notebook or use the bundled example.")
batch = AutoParser(sample_path, n_jobs=1)
rendered = batch.format_transform("xyz", frame=-1, write_to_disk=False)
print(rendered[str(sample_path.resolve())])

3
comment charge 0 multiplicity 1
O               1.7849140000      1.2624220000      0.5119850000
H               2.6482370000      1.0729290000      0.1316310000
H               1.1831680000      1.2568160000     -0.2388350000


The Python API returns a mapping from source path to rendered content when called on a batch. No output file is created by the preview call.

In [2]:
with TemporaryDirectory() as output_dir:
    batch.format_transform(
        "xyz",
        output_dir=output_dir,
        frame=-1,
        write_to_disk=True,
    )
    print(sorted(path.name for path in Path(output_dir).iterdir()))

['water_mp2.xyz']


Gaussian and ORCA input writers accept format-specific options. Review route sections, resources, charge, multiplicity, and solvent settings before submitting generated inputs.

In [3]:
gjf = batch[0][-1].format_transform(
    "gjf",
    route_section="#p B3LYP/6-31G(d) opt",
)
print(gjf.splitlines()[0])
print(next(line for line in gjf.splitlines() if line.startswith("#p")))

%pal
#p B3LYP/6-31G(d) opt
